In [1]:
import os
from datasets import Dataset

img_dir = '/capstor/store/cscs/swissai/infra01/vision-datasets/llava_pretrain/LLaVA-Pretrain/'
image_files = []
for root, dirs, files in os.walk(img_dir):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_files.append(os.path.join(root, f))

dataset = Dataset.from_dict({'file_path': image_files})

print(f"Found {len(image_files)} images.")


/users/rqu/miniconda3/envs/vlm-data/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 558128 images.


In [2]:
from PIL import Image

def get_resolution(batch):
    widths, heights = [], []
    for path in batch['file_path']:
        try:
            with Image.open(path) as img:
                w, h = img.size
        except Exception:
            w, h = None, None
        widths.append(w)
        heights.append(h)
    return {"width": widths, "height": heights}


In [3]:
# Choose a num_proc appropriate for your node (e.g., 32, 48, or 64)
ds = dataset.map(get_resolution, batched=True, batch_size=1024, num_proc=64)

Map (num_proc=64):   0%|          | 0/558128 [00:00<?, ? examples/s]

Map (num_proc=64): 100%|██████████| 558128/558128 [01:36<00:00, 5771.76 examples/s] 


In [4]:
widths = [w for w in ds['width'] if w is not None]
heights = [h for h in ds['height'] if h is not None]


In [5]:
import numpy as np
from collections import Counter

# Assume you already have:
# widths = [ ... ]  # list of int
# heights = [ ... ] # list of int

widths_np = np.array(widths)
heights_np = np.array(heights)

print("=== Image Resolution Statistics ===")
print(f"Number of images: {len(widths)}")

print(f"\nWidth:  min={widths_np.min()}, max={widths_np.max()}, mean={widths_np.mean():.2f}, median={np.median(widths_np)}")
print(f"Height: min={heights_np.min()}, max={heights_np.max()}, mean={heights_np.mean():.2f}, median={np.median(heights_np)}")

# Percentiles
for p in [5, 25, 50, 75, 95]:
    print(f"{p}th percentile width: {np.percentile(widths_np, p):.0f}, height: {np.percentile(heights_np, p):.0f}")

# Aspect ratio stats
ratios = widths_np / heights_np
print(f"\nAspect Ratio (W/H): mean={ratios.mean():.2f}, median={np.median(ratios):.2f}")

# Most common resolutions
res_counter = Counter(zip(widths, heights))
print("\nTop 10 most common resolutions (width x height, count):")
for (w, h), count in res_counter.most_common(10):
    print(f"{w} x {h}: {count}")

# Unique resolutions
print(f"\nNumber of unique resolutions: {len(res_counter)}")


=== Image Resolution Statistics ===
Number of images: 558128

Width:  min=336, max=12192, mean=408.60, median=336.0
Height: min=336, max=5177, mean=368.29, median=336.0
5th percentile width: 336, height: 336
25th percentile width: 336, height: 336
50th percentile width: 336, height: 336
75th percentile width: 462, height: 336
95th percentile width: 597, height: 505

Aspect Ratio (W/H): mean=1.15, median=1.00

Top 10 most common resolutions (width x height, count):
336 x 336: 171011
448 x 336: 50256
504 x 336: 28409
597 x 336: 15706
336 x 504: 14937
336 x 448: 10838
505 x 336: 6504
423 x 336: 5087
503 x 336: 4825
336 x 420: 4323

Number of unique resolutions: 2144


In [6]:
# Given lists: widths, heights, ds (the Hugging Face dataset object)
over_1000 = [(w, h) for w, h in zip(widths, heights) if w is not None and h is not None and (w > 1000 or h > 1000)]
print(f"Number of images with either width or height > 1000: {len(over_1000)}")


Number of images with either width or height > 1000: 1827


In [7]:
import json

json_path = '/capstor/store/cscs/swissai/infra01/vision-datasets/llava_pretrain/LLaVA-Pretrain/blip_laion_cc_sbu_558k.json'
with open(json_path, 'r') as f:
    annotations = json.load(f)

# Build a set of image paths to filter (width/height > 1000)
big_set = set()
for fp, w, h in zip(ds['file_path'], ds['width'], ds['height']):
    if w is not None and h is not None and (w > 1000 or h > 1000):
        # Match how JSON stores paths, e.g., "00453/004539375.jpg"
        base_idx = fp.find('LLaVA-Pretrain/')
        rel_path = fp[base_idx + len('LLaVA-Pretrain/'):] if base_idx >= 0 else fp
        # Remove any leading slashes
        rel_path = rel_path.lstrip('/')
        big_set.add(rel_path)

print(f"Number of images to remove: {len(big_set)}")

# Filter annotations
filtered_annotations = [entry for entry in annotations if entry['image'] not in big_set]
print(f"Number of images after filtering: {len(filtered_annotations)}")

# Save filtered json
filtered_json_path = json_path.replace('.json', '_filtered.json')
with open(filtered_json_path, 'w') as f:
    json.dump(filtered_annotations, f, indent=2)

print(f"Filtered annotation JSON written to {filtered_json_path}")

Number of images to remove: 1827
Number of images after filtering: 556301
Filtered annotation JSON written to /capstor/store/cscs/swissai/infra01/vision-datasets/llava_pretrain/LLaVA-Pretrain/blip_laion_cc_sbu_558k_filtered.json


In [10]:
import json
import numpy as np
from collections import Counter

# Load filtered JSON
filtered_json_path = '/capstor/store/cscs/swissai/infra01/vision-datasets/llava_pretrain/LLaVA-Pretrain/blip_laion_cc_sbu_558k_filtered.json'
with open(filtered_json_path, 'r') as f:
    filtered_annotations = json.load(f)

# Make a set of remaining image relative paths
filtered_image_set = set(entry['image'] for entry in filtered_annotations)

# Build filtered width/height lists using the dataset/filepath info
filtered_widths = []
filtered_heights = []
for fp, w, h in zip(ds['file_path'], ds['width'], ds['height']):
    # Get relative image path as used in JSON (e.g. 00453/004539375.jpg)
    base_idx = fp.find('LLaVA-Pretrain/')
    rel_path = fp[base_idx + len('LLaVA-Pretrain/'):] if base_idx >= 0 else fp
    rel_path = rel_path.lstrip('/')
    if rel_path in filtered_image_set and w is not None and h is not None:
        filtered_widths.append(w)
        filtered_heights.append(h)

# Convert to numpy
widths_np = np.array(filtered_widths)
heights_np = np.array(filtered_heights)

print("=== Image Resolution Statistics (Filtered) ===")
print(f"Number of images: {len(widths_np)}")

print(f"\nWidth:  min={widths_np.min()}, max={widths_np.max()}, mean={widths_np.mean():.2f}, median={np.median(widths_np)}")
print(f"Height: min={heights_np.min()}, max={heights_np.max()}, mean={heights_np.mean():.2f}, median={np.median(heights_np)}")

for p in [5, 25, 50, 75, 95]:
    print(f"{p}th percentile width: {np.percentile(widths_np, p):.0f}, height: {np.percentile(heights_np, p):.0f}")

ratios = widths_np / heights_np
print(f"\nAspect Ratio (W/H): mean={ratios.mean():.2f}, median={np.median(ratios):.2f}")

res_counter = Counter(zip(filtered_widths, filtered_heights))
print("\nTop 10 most common resolutions (width x height, count):")
for (w, h), count in res_counter.most_common(10):
    print(f"{w} x {h}: {count}")

print(f"\nNumber of unique resolutions: {len(res_counter)}")


=== Image Resolution Statistics (Filtered) ===
Number of images: 556301

Width:  min=336, max=1000, mean=407.07, median=336.0
Height: min=336, max=1000, mean=366.92, median=336.0
5th percentile width: 336, height: 336
25th percentile width: 336, height: 336
50th percentile width: 336, height: 336
75th percentile width: 461, height: 336
95th percentile width: 597, height: 504

Aspect Ratio (W/H): mean=1.15, median=1.00

Top 10 most common resolutions (width x height, count):
336 x 336: 171011
448 x 336: 50256
504 x 336: 28409
597 x 336: 15706
336 x 504: 14937
336 x 448: 10838
505 x 336: 6504
423 x 336: 5087
503 x 336: 4825
336 x 420: 4323

Number of unique resolutions: 1313


In [11]:
import json

json_path = "/capstor/store/cscs/swissai/infra01/vision-datasets/llava_pretrain/LLaVA-Pretrain/blip_laion_cc_sbu_558k_filtered.json"
with open(json_path, "r") as f:
    data = json.load(f)

print(f"Number of images in JSON: {len(data)}")


Number of images in JSON: 556301
